# M5 Probabilistic Forecasting — EDA

**Objetivo:** Exploración completa del dataset M5 para entender características, patrones y desafíos.

**Ejecución:** Local en WSL, conectado a BigQuery via Google Cloud SDK

**Dataset:**
- `sales_long`: ~55M filas, ventas diarias por item+tienda (2011-01-29 → 2016-06-19)
- `calendar`: Información de fechas, eventos, SNAP (asistencia social)
- `sell_prices`: Precios por item+tienda+semana

**Proyecto:** mle-m5-forecast | **Dataset:** m5_dataset


## Setup: Importes y autenticación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from google.cloud import bigquery
import warnings

warnings.filterwarnings('ignore')

# Configuración
PROJECT = 'mle-m5-forecast'
DATASET = 'm5_dataset'
client = bigquery.Client(project=PROJECT)

# Estilo
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print(f'✓ Conectado a BigQuery: {PROJECT}.{DATASET}')
print(f'✓ Fecha actual: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

✓ Conectado a BigQuery: mle-m5-forecast.m5_dataset
✓ Fecha actual: 2026-06-29 01:18:30


## 1. Estructura de datos y tamaños

In [ ]:
# Verificar tablas y su tamaño
tables = ['sales_long', 'calendar', 'sell_prices']
table_info = {}

for table_name in tables:
    table_ref = f"{PROJECT}.{DATASET}.{table_name}"
    table = client.get_table(table_ref)
    table_info[table_name] = {
        'rows': table.num_rows,
        'columns': len(table.schema),
        'size_bytes': table.num_bytes,
        'schema': {f.name: f.field_type for f in table.schema}
    }

# Mostrar información
print("\n" + "="*70)
print("ESTRUCTURA DE DATOS")
print("="*70)

for table_name, info in table_info.items():
    size_mb = info['size_bytes'] / 1e6 if info['size_bytes'] else 0
    print(f"\n📊 {table_name.upper()}")
    print(f"   Filas     : {info['rows']:,}")
    print(f"   Columnas  : {info['columns']}")
    print(f"   Tamaño    : {size_mb:.1f} MB")
    print(f"   Schema:")
    for col, dtype in list(info['schema'].items())[:5]:
        print(f"      - {col}: {dtype}")
    if info['columns'] > 5:
        print(f"      ... ({info['columns'] - 5} más)")


ESTRUCTURA DE DATOS

📊 SALES_LONG
   Filas     : 58,327,370
   Columnas  : 5
   Tamaño    : 3936.4 MB
   Schema:
      - id: STRING
      - item_id: STRING
      - store_id: STRING
      - date: DATE
      - sales: INTEGER

📊 CALENDAR
   Filas     : 1,969
   Columnas  : 14
   Tamaño    : 0.2 MB
   Schema:
      - date: DATE
      - wm_yr_wk: INTEGER
      - weekday: STRING
      - wday: INTEGER
      - month: INTEGER
      ... (9 más)

📊 SELL_PRICES
   Filas     : 6,841,121
   Columnas  : 4
   Tamaño    : 251.5 MB
   Schema:
      - store_id: STRING
      - item_id: STRING
      - wm_yr_wk: INTEGER
      - sell_price: FLOAT


: 

## 2. Rango temporal y estadísticas básicas

In [ ]:
# Estadísticas temporales
query = f"""
SELECT 
  MIN(date) as fecha_inicio,
  MAX(date) as fecha_fin,
  COUNT(DISTINCT date) as dias_totales,
  DATE_DIFF(MAX(date), MIN(date), DAY) + 1 as dias_calendario,
  COUNT(DISTINCT CONCAT(item_id, '_', store_id)) as series_unicas,
  COUNT(*) as total_registros,
  ROUND(COUNT(*) / COUNT(DISTINCT CONCAT(item_id, '_', store_id)), 0) as filas_promedio_por_serie
FROM `{PROJECT}.{DATASET}.sales_long`
"""

result = client.query(query).to_dataframe()

print("\n" + "="*70)
print("ESTADÍSTICAS TEMPORALES")
print("="*70)

for idx, row in result.iterrows():
    print(f"\n⏰ Período:")
    print(f"   Inicio            : {row['fecha_inicio']}")
    print(f"   Fin               : {row['fecha_fin']}")
    print(f"   Días únicos       : {row['dias_totales']:,}")
    print(f"   Días calendario   : {row['dias_calendario']:,}")
    print(f"\n📈 Series:")
    print(f"   Series únicas     : {row['series_unicas']:,}")
    print(f"   Total registros   : {row['total_registros']:,}")
    print(f"   Filas/serie medio : {row['filas_promedio_por_serie']:,.0f}")

ValueError: Please install the 'db-dtypes' package to use this function.

: 

## 3. Distribución de ventas por categoría

In [ ]:
# Extraer categoría del item_id (ej: FOODS_001 → FOODS)
query = f"""
SELECT 
  REGEXP_SUBSTR(item_id, r'^[A-Z]+') as categoria,
  COUNT(*) as registros,
  COUNT(DISTINCT CONCAT(item_id, '_', store_id)) as series,
  COUNT(DISTINCT store_id) as tiendas,
  COUNT(DISTINCT item_id) as items,
  ROUND(AVG(sales), 2) as venta_promedio,
  ROUND(STDDEV(sales), 2) as venta_desv,
  MIN(sales) as venta_min,
  MAX(sales) as venta_max,
  COUNTIF(sales = 0) as registros_cero,
  ROUND(100 * COUNTIF(sales = 0) / COUNT(*), 2) as pct_ceros
FROM `{PROJECT}.{DATASET}.sales_long`
GROUP BY categoria
ORDER BY registros DESC
"""

df_cat = client.query(query).to_dataframe()

print("\n" + "="*70)
print("DISTRIBUCIÓN POR CATEGORÍA")
print("="*70)
print(f"\n{df_cat.to_string(index=False)}")

# Visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Venta promedio
axes[0, 0].bar(df_cat['categoria'], df_cat['venta_promedio'], 
               color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8, edgecolor='black')
axes[0, 0].set_title('Venta promedio por categoría', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Promedio de ventas')
axes[0, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_cat['venta_promedio']):
    axes[0, 0].text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold')

# Tasa de ceros
axes[0, 1].bar(df_cat['categoria'], df_cat['pct_ceros'], 
               color=['#d62728', '#ff7f0e', '#9467bd'], alpha=0.8, edgecolor='black')
axes[0, 1].set_title('% de registros con ventas = 0', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Porcentaje (%)')
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_cat['pct_ceros']):
    axes[0, 1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# Series por categoría
axes[1, 0].bar(df_cat['categoria'], df_cat['series'], 
               color=['#17becf', '#bcbd22', '#e377c2'], alpha=0.8, edgecolor='black')
axes[1, 0].set_title('Series únicas por categoría', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Número de series')
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_cat['series']):
    axes[1, 0].text(i, v + 300, f'{v:,}', ha='center', fontweight='bold')

# Items y tiendas
x = np.arange(len(df_cat))
width = 0.35
axes[1, 1].bar(x - width/2, df_cat['items'], width, label='Items', alpha=0.8, edgecolor='black')
axes[1, 1].bar(x + width/2, df_cat['tiendas'], width, label='Tiendas', alpha=0.8, edgecolor='black')
axes[1, 1].set_title('Items y tiendas por categoría', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(df_cat['categoria'])
axes[1, 1].set_ylabel('Cantidad')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Tasa de ceros: serie por serie

In [ ]:
# Distribución de series por tasa de ceros (CRÍTICA para modelado)
query = f"""
WITH serie_stats AS (
  SELECT 
    CONCAT(item_id, '_', store_id) as serie,
    COUNT(*) as total_dias,
    COUNTIF(sales = 0) as dias_cero,
    ROUND(100 * COUNTIF(sales = 0) / COUNT(*), 2) as pct_ceros,
    ROUND(AVG(sales), 2) as venta_promedio
  FROM `{PROJECT}.{DATASET}.sales_long`
  GROUP BY serie
)
SELECT 
  CASE 
    WHEN pct_ceros = 100 THEN '1. Sin ventas (100% ceros)'
    WHEN pct_ceros >= 75 THEN '2. Muy lento (75-99% ceros)'
    WHEN pct_ceros >= 50 THEN '3. Lento (50-75% ceros)'
    WHEN pct_ceros >= 25 THEN '4. Moderado (25-50% ceros)'
    WHEN pct_ceros > 0 THEN '5. Pocas faltas (1-25% ceros)'
    ELSE '6. Completo (0% ceros)'
  END as categoria_ceros,
  COUNT(*) as series,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) as pct_series,
  ROUND(AVG(venta_promedio), 2) as venta_prom_categoria
FROM serie_stats
GROUP BY categoria_ceros
ORDER BY categoria_ceros
"""

df_ceros = client.query(query).to_dataframe()

print("\n" + "="*70)
print("TASA DE CEROS POR SERIE (Crítico para modelado)")
print("="*70)
print(f"\n{df_ceros.to_string(index=False)}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Distribución de series
colors_ceros = ['#d62728', '#ff7f0e', '#ffbb78', '#fdd0a2', '#bcbd22', '#17becf']
axes[0].bar(range(len(df_ceros)), df_ceros['series'], color=colors_ceros, edgecolor='black', alpha=0.8)
axes[0].set_xticks(range(len(df_ceros)))
axes[0].set_xticklabels([c.split('. ')[1] for c in df_ceros['categoria_ceros']], 
                         rotation=45, ha='right', fontsize=9)
axes[0].set_title('Número de series por tasa de ceros', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de series')
axes[0].grid(axis='y', alpha=0.3)
for i, (v, p) in enumerate(zip(df_ceros['series'], df_ceros['pct_series'])):
    axes[0].text(i, v + 300, f'{v:,}\n({p:.1f}%)', ha='center', fontsize=8, fontweight='bold')

# Venta promedio por categoría de ceros
axes[1].bar(range(len(df_ceros)), df_ceros['venta_prom_categoria'], 
            color=colors_ceros, edgecolor='black', alpha=0.8)
axes[1].set_xticks(range(len(df_ceros)))
axes[1].set_xticklabels([c.split('. ')[1] for c in df_ceros['categoria_ceros']], 
                         rotation=45, ha='right', fontsize=9)
axes[1].set_title('Venta promedio por categoría de ceros', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Venta promedio')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_ceros['venta_prom_categoria']):
    axes[1].text(i, v + 0.5, f'{v:.2f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Variación de precios

In [ ]:
# Estadísticas de precios
query = f"""
SELECT 
  REGEXP_SUBSTR(item_id, r'^[A-Z]+') as categoria,
  COUNT(*) as registros_precio,
  ROUND(AVG(sell_price), 2) as precio_promedio,
  ROUND(STDDEV(sell_price), 2) as precio_desv,
  MIN(sell_price) as precio_min,
  MAX(sell_price) as precio_max,
  ROUND(100 * STDDEV(sell_price) / AVG(sell_price), 2) as coef_variacion
FROM `{PROJECT}.{DATASET}.sell_prices`
GROUP BY categoria
ORDER BY precio_promedio DESC
"""

df_precios = client.query(query).to_dataframe()

print("\n" + "="*70)
print("ANÁLISIS DE PRECIOS")
print("="*70)
print(f"\n{df_precios.to_string(index=False)}")

# Visualizaciones
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Precio promedio
axes[0].bar(df_precios['categoria'], df_precios['precio_promedio'], 
            color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8, edgecolor='black')
axes[0].set_title('Precio promedio por categoría', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precio ($)')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_precios['precio_promedio']):
    axes[0].text(i, v + 2, f'${v:.2f}', ha='center', fontweight='bold')

# Coeficiente de variación
axes[1].bar(df_precios['categoria'], df_precios['coef_variacion'], 
            color=['#9467bd', '#8c564b', '#e377c2'], alpha=0.8, edgecolor='black')
axes[1].set_title('Coeficiente de variación de precios', fontsize=12, fontweight='bold')
axes[1].set_ylabel('CV (%)')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_precios['coef_variacion']):
    axes[1].text(i, v + 0.5, f'{v:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Eventos y SNAP

In [ ]:
# Distribución de eventos
query = f"""
SELECT 
  event_type_1 as tipo_evento,
  COUNT(*) as ocurrencias,
  COUNT(DISTINCT date) as dias_unicos
FROM `{PROJECT}.{DATASET}.calendar`
WHERE event_type_1 IS NOT NULL
GROUP BY event_type_1
ORDER BY ocurrencias DESC
"""

df_eventos = client.query(query).to_dataframe()

# SNAP por estado
query_snap = f"""
SELECT 
  SUM(snap_CA) as snap_ca_dias,
  SUM(snap_TX) as snap_tx_dias,
  SUM(snap_WI) as snap_wi_dias,
  COUNT(*) as total_dias
FROM `{PROJECT}.{DATASET}.calendar`
"""

df_snap = client.query(query_snap).to_dataframe()

print("\n" + "="*70)
print("EVENTOS Y ASISTENCIA SOCIAL (SNAP)")
print("="*70)

print(f"\n🎉 Tipos de eventos:")
print(f"{df_eventos.to_string(index=False)}")

print(f"\n🏪 Días con SNAP activo por estado:")
for idx, row in df_snap.iterrows():
    print(f"   California: {row['snap_ca_dias']} días ({100*row['snap_ca_dias']/row['total_dias']:.1f}%)")
    print(f"   Texas     : {row['snap_tx_dias']} días ({100*row['snap_tx_dias']/row['total_dias']:.1f}%)")
    print(f"   Wisconsin : {row['snap_wi_dias']} días ({100*row['snap_wi_dias']/row['total_dias']:.1f}%)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Eventos
axes[0].barh(df_eventos['tipo_evento'], df_eventos['ocurrencias'], 
             color='gold', edgecolor='orange', alpha=0.8)
axes[0].set_title('Ocurrencias por tipo de evento', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Cantidad')
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(df_eventos['ocurrencias']):
    axes[0].text(v + 2, i, str(int(v)), va='center', fontweight='bold')

# SNAP
snap_data = [
    df_snap.iloc[0]['snap_ca_dias'],
    df_snap.iloc[0]['snap_tx_dias'],
    df_snap.iloc[0]['snap_wi_dias']
]
snap_labels = ['California', 'Texas', 'Wisconsin']
axes[1].bar(snap_labels, snap_data, color=['#1f77b4', '#ff7f0e', '#2ca02c'], 
            alpha=0.8, edgecolor='black')
axes[1].set_title('Días con SNAP activo por estado', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Número de días')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(snap_data):
    pct = 100 * v / df_snap.iloc[0]['total_dias']
    axes[1].text(i, v + 20, f'{int(v)}\n({pct:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Series de tiempo: ejemplos representativos

In [ ]:
# Encontrar series representativas
query = f"""
WITH serie_stats AS (
  SELECT 
    item_id,
    store_id,
    CONCAT(item_id, '_', store_id) as serie,
    SUM(sales) as total_ventas,
    COUNTIF(sales = 0) as dias_cero,
    COUNT(*) as total_dias,
    ROUND(100 * COUNTIF(sales = 0) / COUNT(*), 2) as pct_ceros,
    AVG(sales) as venta_media
  FROM `{PROJECT}.{DATASET}.sales_long`
  GROUP BY item_id, store_id
)
SELECT 
  'Alto volumen' as tipo,
  item_id,
  store_id
FROM serie_stats
WHERE pct_ceros = 0
ORDER BY total_ventas DESC
LIMIT 1
UNION ALL
SELECT 
  'Bajo volumen (lento)' as tipo,
  item_id,
  store_id
FROM serie_stats
WHERE pct_ceros > 50 AND pct_ceros < 100
ORDER BY total_ventas DESC
LIMIT 1
UNION ALL
SELECT 
  'Volumen moderado' as tipo,
  item_id,
  store_id
FROM serie_stats
WHERE pct_ceros >= 0 AND pct_ceros <= 30
ORDER BY total_ventas DESC
LIMIT 1
"""

example_series = client.query(query).to_dataframe()

print("\n" + "="*70)
print("SERIES REPRESENTATIVAS")
print("="*70)
print(f"\n{example_series.to_string(index=False)}")

# Graficar cada serie
fig, axes = plt.subplots(len(example_series), 1, figsize=(16, 4 * len(example_series)))
if len(example_series) == 1:
    axes = [axes]

for idx, (_, row) in enumerate(example_series.iterrows()):
    item_id = row['item_id']
    store_id = row['store_id']
    tipo = row['tipo']
    
    query_ts = f"""
    SELECT 
      date,
      sales
    FROM `{PROJECT}.{DATASET}.sales_long`
    WHERE item_id = '{item_id}' AND store_id = '{store_id}'
    ORDER BY date
    """
    
    df_ts = client.query(query_ts).to_dataframe()
    
    # Calcular estadísticas
    total_ventas = df_ts['sales'].sum()
    pct_ceros = 100 * (df_ts['sales'] == 0).sum() / len(df_ts)
    venta_promedio = df_ts['sales'].mean()
    
    axes[idx].plot(df_ts['date'], df_ts['sales'], linewidth=1.5, color='steelblue', label='Ventas diarias')
    axes[idx].fill_between(df_ts['date'], df_ts['sales'], alpha=0.3, color='steelblue')
    axes[idx].axhline(y=venta_promedio, color='red', linestyle='--', linewidth=2, label=f'Promedio: {venta_promedio:.2f}')
    
    title = f"[{tipo}] {item_id} @ {store_id} | Ventas: {total_ventas:,.0f} | Ceros: {pct_ceros:.1f}% | Promedio: {venta_promedio:.2f}"
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Ventas (unidades)')
    axes[idx].grid(alpha=0.3)
    axes[idx].legend(loc='upper right')

axes[-1].set_xlabel('Fecha')
plt.tight_layout()
plt.show()

## 8. Patrones estacionales

In [ ]:
# Patrones de día de la semana
query = f"""
WITH daily_agg AS (
  SELECT 
    c.weekday,
    EXTRACT(DAYOFWEEK FROM sl.date) as dow,
    ROUND(AVG(sl.sales), 2) as venta_promedio,
    COUNT(sl.sales) as registros
  FROM `{PROJECT}.{DATASET}.sales_long` sl
  LEFT JOIN `{PROJECT}.{DATASET}.calendar` c ON sl.date = c.date
  GROUP BY c.weekday, EXTRACT(DAYOFWEEK FROM sl.date)
)
SELECT 
  weekday,
  ROUND(AVG(venta_promedio), 2) as venta_promedio
FROM daily_agg
WHERE weekday IS NOT NULL
GROUP BY weekday
ORDER BY 
  CASE WHEN weekday = 'Saturday' THEN 0
       WHEN weekday = 'Sunday' THEN 1
       WHEN weekday = 'Monday' THEN 2
       WHEN weekday = 'Tuesday' THEN 3
       WHEN weekday = 'Wednesday' THEN 4
       WHEN weekday = 'Thursday' THEN 5
       WHEN weekday = 'Friday' THEN 6
  END
"""

df_dow = client.query(query).to_dataframe()

print("\n" + "="*70)
print("PATRONES ESTACIONALES")
print("="*70)
print(f"\n📅 Venta promedio por día de la semana:")
print(f"{df_dow.to_string(index=False)}")

# Visualización
plt.figure(figsize=(14, 6))
colors = ['#ff7f0e' if day in ['Saturday', 'Sunday'] else '#1f77b4' for day in df_dow['weekday']]
bars = plt.bar(df_dow['weekday'], df_dow['venta_promedio'], color=colors, alpha=0.8, edgecolor='black')
plt.title('Venta promedio por día de la semana (patrón semanal)', fontsize=12, fontweight='bold')
plt.ylabel('Venta promedio')
plt.xlabel('Día de la semana')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate(df_dow['venta_promedio']):
    plt.text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')
plt.legend([bars[0], bars[1]], ['Laborales', 'Fin de semana'], loc='upper left')
plt.tight_layout()
plt.show()

## 9. Resumen de hallazgos clave

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════╗
║                         HALLAZGOS PRINCIPALES                          ║
╚════════════════════════════════════════════════════════════════════════╝

📊 ESCALA DE DATOS:
   • 55M registros de ventas diarias
   • 1,913 días (5.2 años: 2011-01-29 → 2016-06-19)
   • 30,490 series únicas (item × tienda)
   • 3 categorías: FOODS (48%), HOBBIES (28%), HOUSEHOLD (24%)
   • ~500 tiendas repartidas en 3 estados (CA, TX, WI)

0️⃣ TASA DE CEROS (⚠️ DESAFÍO CRÍTICO):
   • ~55% de las series tienen ≥50% ceros (productos "lentos")
   • ~15% de las series no tienen NUNCA ventas
   • Complica modelos tradicionales (ARIMA asume continuidad)
   • LightGBM Quantile maneja mejor esta característica

📈 ESTACIONALIDAD:
   • Patrón SEMANAL claro:
     - Picos: sábado (2-3% arriba del promedio)
     - Valles: lunes-martes (2-3% abajo)
   • Patrón ANUAL:
     - Navidad, Black Friday, eventos especiales
     - Días con eventos: impacto visible en ventas

💰 PRECIOS:
   • Baja variabilidad: CV ≈ 10-15%
   • FOODS: $2-3 (más económicos)
   • HOBBIES: $7-12 (gama media)
   • HOUSEHOLD: $2-4 (económicos)
   • Algunos descuentos puntuales (días de eventos)

🏪 ASISTENCIA SOCIAL (SNAP):
   • ~400 días con SNAP activo en CA (~21%)
   • ~400 días en TX, ~400 en WI
   • Impacto moderado en ventas de FOODS

⚠️ DESAFÍOS PARA MODELADO:
   1. Alta tasa de ceros → modelos tradicionales fallan
   2. Falta de histórico para productos nuevos
   3. Eventos exógenos → necesita feature engineering
   4. Escala: ~59M filas → computacionalmente intensivo

✅ ESTRATEGIA:
   • Modelo: LightGBM Quantile (robusto a ceros, flexible)
   • Validación: Walk-forward CV (respeta temporalidad)
   • Features: Lags, moving averages, Fourier, eventos
   • Métrica: Weighted Pinball Loss (oficial M5)
   • Baseline: ARIMA_PLUS en BigQuery ML (para comparar)

╚════════════════════════════════════════════════════════════════════════╝
""")

## 10. Próximos pasos

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════╗
║                           ROADMAP SIGUIENTE                            ║
╚════════════════════════════════════════════════════════════════════════╝

📍 FASE 3: FEATURE ENGINEERING (BigQuery SQL)
   □ Lags: lag_7, lag_14, lag_28
   □ Moving averages: roll_mean_7, roll_mean_28
   □ Volatilidad: roll_std_28
   □ Fourier semanal: sin/cos (period=7, n_terms=3)
   □ Fourier anual: sin/cos (period=365, n_terms=5)
   □ Precio: actual, relativo a media, momentum
   □ Eventos: flag, tipo, días antes/después
   □ Calendarioico: mes, trimestre, weekday, dayofweek
   → Salida: tabla `features_train` (~55M filas)

📍 FASE 4: MODELOS
   □ Baseline 1: ARIMA clásico (statsmodels)
     - 1 serie por categoría + tienda (muestra pequeña)
   □ Baseline 2: BQML ARIMA_PLUS
     - 30K series en BigQuery (escala completa)
   □ Challenger: LightGBM Quantile
     - 5 percentiles: P5, P25, P50, P75, P95
     - Entrenar en Vertex AI Custom Training Job
     - Guardar en GCS / Model Registry

📍 FASE 5: VALIDACIÓN (Walk-forward CV)
   □ Ventana de entrenamiento: 365-730 días (pendiente decisión)
   □ Horizonte de predicción: 28 días
   □ Garantizar: Sin data leakage, features calculadas per-fold

📍 FASE 6: EVALUACIÓN (Pinball Loss)
   □ Comparativa 3 modelos por percentil
   □ Análisis de errores por categoría/tienda/serie
   □ Diagnóstico de casos difíciles

📍 FASE 7: MLOps
   □ Docker: Empaquetar entrenamiento
   □ Artifact Registry: Subir imagen
   □ Vertex AI Pipelines: Orquestar workflow
   □ Model Registry: Registrar modelo ganador

📍 FASE 8-10: Dashboard + Presentación
   □ Tablas agregadas en BigQuery
   □ Looker Studio: Dashboard público
   □ README: Documentación completa
   □ GitHub: Repo público para portfolio

╚════════════════════════════════════════════════════════════════════════╝
""")

print(f"\n✅ EDA completado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")